In [1]:
pip install panda

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
  Running setup.py install for panda: started
  Running setup.py install for panda: finished with status 'done'
Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: panda is being installed using the legacy 'setup.py install' method, because it does not have a 'pyproject.toml' and the 'wheel' package is not installed. pip 23.1 will enforce this behaviour change. A possible replacement is to enable the '--use-pep517' option. Discussion can be found at https://github.com/pypa/pip/issues/8559

[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
pip install bs4

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [20]:
from bs4 import BeautifulSoup
import pandas as pd
import os

d = {'Date': [], 'Headline': [], 'Link': []}
file = "sypnlnewshtml" + ".html"
# for file in os.listdir("STOCKNEPSE"):
try:
    with open(f"STOCKNEPSE/webScraphtmlfiles/{file}", encoding="utf-8") as f:
        html_doc = f.read()
    
    soup = BeautifulSoup(html_doc, 'html.parser')        
    rows = soup.find('tbody').find_all('tr')
    
    for row in rows:
        cols = row.find_all('td')
        if len(cols) >= 1:
            link_tag = cols[1].find('a')
            d['Date'].append(cols[0].get_text().strip())
            d['Headline'].append(cols[1].get_text().strip())
            d['Link'].append(link_tag['href'].strip())
            # d['Full News'].append(cols[4].get_text().strip())
    
    print(f"{file}: {len(rows)} rows")
    
except Exception as e:
    print(f"{e}")
csv_filename = file.replace('.html', '.csv')
df = pd.DataFrame(data=d)
df.to_csv(f"STOCKNEPSE/NEPSEDATA/{csv_filename}", index=False)

sypnlnewshtml.html: 15 rows


In [26]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time

# Read the CSV file
csv_filename = "sypnlnewshtml.csv"
df = pd.read_csv(f"STOCKNEPSE/NEPSEDATA/{csv_filename}")

print(f"Total news items: {len(df)}\n")

# Create a new dictionary to store full news content
news_data = {'Date': [], 'Headline': [], 'Link': [], 'Full Content': []}

for index, row in df.iterrows():
    date = row['Date']
    headline = row['Headline']
    link = row['Link']
    
    print(f"\nProcessing {index + 1}/{len(df)}: {headline}")
    
    try:
        # Make request to the link
        response = requests.get(link, timeout=10)
        response.raise_for_status()
        
        # Parse HTML
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Find the newsdetail-content div
        content_div = soup.find('div', id='newsdetail-content')
        
        if content_div:
            # Get all text from the div
            content = content_div.get_text(separator='\n', strip=True)
            print(f"Extracted {len(content)} characters")
        else:
            content = 'Content div not found'
            print("Warning: newsdetail-content div not found")
        
        # Store data
        news_data['Date'].append(date)
        news_data['Headline'].append(headline)
        news_data['Link'].append(link)
        news_data['Full Content'].append(content)
        
    except Exception as e:
        print(f"Error: {e}")
        news_data['Date'].append(date)
        news_data['Headline'].append(headline)
        news_data['Link'].append(link)
        news_data['Full Content'].append(f'Error: {str(e)}')
    
    time.sleep(1)  # Be polite to the server

# Save to new CSV with full content
output_filename = "sypnlnews_full_content.csv"
df_full = pd.DataFrame(news_data)
df_full.to_csv(f"STOCKNEPSE/NEPSEDATA/{output_filename}", index=False)

print(f"\n{'='*80}")
print(f"Saved full content to: STOCKNEPSE/NEPSEDATA/{output_filename}")
print(f"Total articles processed: {len(df_full)}")

# Display summary
print(f"\nSuccessfully extracted: {len([c for c in news_data['Full Content'] if not c.startswith('Error')])}")
print(f"Errors: {len([c for c in news_data['Full Content'] if c.startswith('Error')])}")

Total news items: 15


Processing 1/15: IPO shares of SY Panel Nepal Limited Listed in NEPSE; What is the Opening Ranges?
Extracted 623 characters

Processing 2/15: SY Panel Nepal Limited IPO Allotment Concludes: Lucky 4,07,615 Applicants Allotted 10 Units Each via Lottery System
Extracted 2257 characters

Processing 3/15: IPO Issue of SY Panel Nepal Limited Closing Today; Oversubscribed 4.80 Times So Far
Extracted 2076 characters

Processing 4/15: IPO for General Public: SY Panel Nepal Limited Company Issue 40,76,156 Units IPO Shares from Today
Extracted 2152 characters

Processing 5/15: IPO for General Public: SY Panel Nepal Limited to Issue 40,76,156 Units IPO Shares from Kartik 19
Extracted 2141 characters

Processing 6/15: SY Panel Nepal Limited Closing IPO Shares to Project-Affected Locals of Chitwan From Today
Extracted 1776 characters

Processing 7/15: SY Panel Nepal Limited Closing 4,97,092 Units IPO Shares to Foreign Nepalese Immigrants From Today
Extracted 1948 characters

P